<a href="https://colab.research.google.com/github/MathMachado/DSWP/blob/master/Notebooks/GAM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# GAM

Olá! Que prazer ver alguém mergulhando nos **Modelos Aditivos Generalizados (GAMs)**. Como estatístico, posso te dizer: se os GLMs (Modelos Lineares Generalizados) são uma ferramenta de precisão, os GAMs são o canivete suíço de alta performance. Eles permitem capturar relações não lineares sem que você precise "adivinhar" a forma da curva (se é quadrática, logarítmica, etc.).

Vamos estruturar seu estudo para o concurso focando no que realmente cai nas provas de alto nível.

---

## 1. Estrutura do Modelo: Do GLM ao GAM

A grande sacada do GAM é substituir o preditor linear $\sum \beta_i x_i$ por uma soma de funções suaves $f_i(x_i)$.

### A Equação Matemática
A forma geral de um GAM é definida como:

$$g(E(Y)) = \beta_0 + f_1(x_1) + f_2(x_2) + \dots + f_p(x_p)$$

Onde:
* **$g(\cdot)$**: É a **função de ligação** (link function), como nos GLMs (ex: logit para Binomial, log para Poisson).
* **$f_i(x_i)$**: São as **funções de suavização** (smooth functions) que assumem formas flexíveis.

---

## 2. Funções de Suavização (Smoothing) e Bases

Para que o computador consiga estimar uma "curva", nós a decompomos em funções menores chamadas **Funções de Base**.

* **Splines de Regressão:** É a técnica mais comum. Imagine dividir o eixo $X$ em vários segmentos (através de pontos chamados **nós** ou *knots*). Em cada segmento, ajustamos um polinômio (geralmente de 3º grau) que se conecta suavemente nos nós.
* **Bases Comuns:** Splines cúbicas, *Thin plate regression splines* (padrão no R) e *P-splines*.



---

## 3. Penalização e Ajuste (O "Trade-off")

Aqui é onde os concursos costumam apertar. Se usarmos muitos nós, o modelo sofre **overfitting** (decora o ruído). Se usarmos poucos, sofremos **underfitting**.

### A Verossimilhança Penalizada
O GAM resolve isso minimizando uma função que soma o erro e uma penalidade de "rugosidade":

$$\min \left\{ \text{Deviance} + \lambda \int [f''(x)]^2 dx \right\}$$

* **$\lambda$ (Parâmetro de Suavização):** Controla o equilíbrio.
    * Se $\lambda \to \infty$: A curva vira uma **reta** (penalidade máxima).
    * Se $\lambda = 0$: A curva vira uma **interpolação perfeita** (sem penalidade).
* **Ajuste:** Geralmente feito via **REML** (Restricted Maximum Likelihood) ou **GCV** (Generalized Cross Validation) para encontrar o $\lambda$ ótimo automaticamente.

---

## 4. Diagnóstico e Validação

Diferente da regressão linear simples, aqui olhamos para a "curvatura":

1.  **EDF (Estimated Degrees of Freedom):** Representa a complexidade da curva.
    * $EDF = 1$: Relação linear.
    * $EDF > 1$: Relação não linear (quanto maior, mais "ondulada" é a curva).
2.  **Verificação dos Resíduos:** Assim como no GLM, analisamos os resíduos de deviance para checar heterocedasticidade e normalidade.
3.  **Convergência:** Verificar se o algoritmo de ajuste convergiu (problemas comuns em dados esparsos).

---

## 5. Prática: O Fluxo no R (Pacote `mgcv`)

O pacote padrão-ouro para GAM é o `mgcv`, desenvolvido por Simon Wood.

```r
# Instalação e carga
library(mgcv)

# Ajuste do Modelo
# s(x) indica que queremos uma função de suavização para a variável x
modelo_gam <- gam(y ~ s(x1) + s(x2) + x3,
                  family = poisson(),
                  data = seus_dados,
                  method = "REML")

# Resumo estatístico
summary(modelo_gam)

# Diagnóstico visual
gam.check(modelo_gam)

# Visualização das curvas
plot(modelo_gam, pages = 1, residuals = TRUE, shade = TRUE)
```

---

## Dicas de Ouro para a Prova

* **Interpretabilidade:** O GAM é considerado um modelo "semi-paramétrico". Ele é mais interpretável que uma Redeneural, pois você ainda consegue ver o efeito isolado de cada variável no gráfico de `plot()`.
* **Maldição da Dimensionalidade:** Os GAMs ajudam a combatê-la porque tratam cada variável de forma aditiva, evitando que o número de parâmetros exploda como em interações polinomiais complexas.
* **Diferença GAM vs GLM:** O GLM assume que você sabe a forma funcional (linear nos parâmetros); o GAM aprende a forma funcional a partir dos dados.

Você gostaria que eu aprofundasse em algum tópico específico, como o cálculo da **Deviance** ou como escolher entre **GCV e REML**?

Excelente escolha. Esses são os pontos que separam quem apenas "roda o código" de quem realmente entende a teoria estatística por trás dos modelos—exatamente o que bancas de concurso para professor buscam.

---

## 1. GLM vs. GAM: A Flexibilidade do Preditor

A principal diferença não está na distribuição dos dados (ambos aceitam a Família Exponencial), mas na forma do **preditor linear**.

| Característica | GLM (Modelos Lineares Generalizados) | GAM (Modelos Aditivos Generalizados) |
| :--- | :--- | :--- |
| **Preditor** | $\eta = \sum \beta_i x_i$ | $\eta = \sum f_i(x_i)$ |
| **Relação** | Assume linearidade nos parâmetros. | Apreme relações não lineares via *splines*. |
| **Flexibilidade** | Rígido. Exige transformação manual de $X$ (ex: $x^2$). | Flexível. A curva "segue" os dados onde há sinal. |
| **Risco** | Maior risco de viés (*underfitting*) se a forma for errada. | Maior risco de variância (*overfitting*) se não penalizado. |



---

## 2. Aprofundando no Cálculo da Deviance

A **Deviance** é o análogo da "Soma dos Quadrados dos Resíduos" para modelos generalizados. Ela mede a qualidade do ajuste comparando o seu modelo com um **Modelo Saturado** (um modelo hipotético que tem um parâmetro para cada observação e ajusta os dados perfeitamente).

### A Fórmula
A Deviance ($D$) é definida como:
$$D = 2 \phi [ \ell(\hat{\beta}_{sat}; y) - \ell(\hat{\beta}_{mod}; y) ]$$

Onde:
* **$\ell(\hat{\beta}_{sat}; y)$**: Log-verossimilhança do modelo saturado.
* **$\ell(\hat{\beta}_{mod}; y)$**: Log-verossimilhança do seu modelo atual.
* **$\phi$**: Parâmetro de dispersão.

**Por que ela importa no GAM?**
No ajuste do GAM, não minimizamos apenas a Deviance, mas a **Deviance Penalizada**:
$$\mathcal{P} = D(\beta) + \sum \lambda_j \int [f''_j(x_j)]^2 dx_j$$
Isso garante que o modelo seja preciso (baixa deviance), mas sem oscilações erráticas (baixa penalidade).

---

## 3. GCV vs. REML: Qual critério escolher?

Para ajustar um GAM, precisamos escolher o valor de $\lambda$ (o "freio" da suavização). Existem dois métodos principais:

### GCV (Generalized Cross-Validation)
Historicamente muito popular, o GCV tenta minimizar o erro de previsão extra-amostral de forma aproximada, sem precisar de validação cruzada real (que seria computacionalmente cara).
* **Vantagem:** Rápido e não assume uma distribuição formal para os coeficientes de suavização.
* **Desvantagem:** Tende a ser instável e tem uma propensão matemática ao **overfitting** (escolher modelos mais complexos do que o necessário), especialmente em amostras pequenas.

### REML (Restricted Maximum Likelihood)
O REML trata as funções de suavização como **efeitos aleatórios**. Ele estima $\lambda$ maximizando a parte da verossimilhança que não depende dos efeitos fixos.
* **Vantagem:** É muito mais robusto e menos propenso a subestimar a penalidade (evita overfitting). É o padrão recomendado atualmente (especialmente por Simon Wood, autor do `mgcv`).
* **Desvantagem:** Levemente mais exigente computacionalmente que o GCV.

> **Dica para a prova:** Se a questão perguntar qual o método de escolha moderno para evitar overfitting e garantir convergência estável, a resposta é **REML**.

---

## 4. Resumo Prático de Diagnóstico

Ao olhar para o `summary(modelo_gam)`, foque nestes dois indicadores para validar o ajuste:

1.  **P-valor dos termos suaves:** Testa a hipótese nula de que $f(x) = 0$. Se for significativo, a variável contribui para o modelo.
2.  **EDF (Graus de Liberdade Estimados):** * $EDF \approx 1$: A relação é linear. Você poderia usar um GLM comum.
    * $EDF > 2$: Existe uma curvatura clara.
    * $EDF$ próximo ao limite superior ($k-1$): Sinal de que você precisa aumentar o número de nós (*basis functions*), pois a curva pode estar "presa".

Ficou clara a diferença de como a Deviance atua na função objetivo do GAM em comparação com o GLM tradicional?